# NumPy Session 2: The Power Tools
## Aggregations, Broadcasting & Boolean Masking

---

### What You'll Learn in This Session

By the end of this session, you will be able to:

1. **Understand UFuncs** — Why vectorized operations are 100x faster than loops
2. **Aggregate data** — Calculate sum, mean, min, max along rows/columns
3. **Master Broadcasting** — Perform math on arrays of different shapes
4. **Filter data with Boolean Masks** — "Give me all rows where X > 5"

---

###  Why These Topics Matter for Data Science

| Topic | Real-World Application |
|-------|------------------------|
| **Aggregations** | Calculate average salary by department, find max temperature per city |
| **Broadcasting** | Normalize features (subtract mean, divide by std) — required for ML |
| **Boolean Masking** | Filter outliers, select data subsets, handle missing values |

> **Real Talk**: These three concepts are what you'll use every single day in EDA (Exploratory Data Analysis). Master them, and Pandas becomes trivial.

---

## Chapter 1: Universal Functions (UFuncs) — Speed Matters

###  Why Should I Care?

In data science, you'll work with **millions** of data points. The difference between a 5-second operation and a 0.05-second operation is the difference between productive work and frustration.

**The Golden Rule**: Never use Python loops on NumPy arrays. Always use vectorized operations.

### 1.1 The Slowness of Loops

In [1]:
import numpy as np
rng = np.random.default_rng(seed=1701)

# The WRONG way: Python loop
def compute_reciprocals_slow(values):
    output = np.empty(len(values))
    for i in range(len(values)):
        output[i] = 1.0 / values[i]
    return output

values = rng.integers(1, 10, size=5)
print(f"Input: {values}")
print(f"Output: {compute_reciprocals_slow(values)}")

Input: [9 4 1 3 8]
Output: [0.11111111 0.25       1.         0.33333333 0.125     ]


In [2]:
# Let's time it with 1 million elements
big_array = rng.integers(1, 100, size=1_000_000)

print("Timing Python loop (this will take several seconds)...")
%timeit compute_reciprocals_slow(big_array)

Timing Python loop (this will take several seconds)...
4.14 s ± 385 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


11.4 s ± 1.79 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


### 1.2 The UFuncs Solution: Vectorized Operations

**Vectorization** = Apply operation to entire array at once, not element-by-element

```
Loop (slow):       for i in range(len(arr)): result[i] = 1/arr[i]
Vectorized (fast): result = 1/arr
```

```
┌─────────────────────────────────────────────────────────────┐
│ Python Loop: Process one element at a time                  │
│ ┌───┐   ┌───┐   ┌───┐   ┌───┐   ┌───┐                      │
│ │ 1 │ → │ 2 │ → │ 3 │ → │ 4 │ → │ 5 │  (sequential)        │
│ └───┘   └───┘   └───┘   └───┘   └───┘                      │
├─────────────────────────────────────────────────────────────┤
│ NumPy UFuncs: Process ALL elements simultaneously           │
│ ┌───┬───┬───┬───┬───┐                                      │
│ │ 1 │ 2 │ 3 │ 4 │ 5 │  ──→ ALL AT ONCE (parallel)         │
│ └───┴───┴───┴───┴───┘                                      │
└─────────────────────────────────────────────────────────────┘
```

In [3]:
# The RIGHT way: Vectorized operation
print("Old way (loop):")
print(compute_reciprocals_slow(values))

print("\nNew way (vectorized):")
print(1.0 / values)  # Same result, WAY faster!

Old way (loop):
[0.11111111 0.25       1.         0.33333333 0.125     ]

New way (vectorized):
[0.11111111 0.25       1.         0.33333333 0.125     ]


In [4]:
# Time the vectorized version
print("Timing vectorized operation...")
%timeit (1.0 / big_array)

Timing vectorized operation...
6.32 ms ± 140 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### no Speed Comparison Summary

| Method | Time for 1M elements | Speedup |
|--------|---------------------|--------|
| Python loop | ~5 seconds | 1x |
| NumPy ufunc | ~5 milliseconds | **~1000x** |

### 1.3 Common UFuncs: Your New Best Friends

In [5]:
x = np.array([1, 2, 3, 4, 5])
print(f"x = {x}")
print()
print("Arithmetic UFuncs:")
print(f"x + 5  = {x + 5}")
print(f"x - 2  = {x - 2}")
print(f"x * 3  = {x * 3}")
print(f"x / 2  = {x / 2}")
print(f"x // 2 = {x // 2}  ← floor division")
print(f"x ** 2 = {x ** 2}  ← power")
print(f"x % 2  = {x % 2}   ← modulo")

x = [1 2 3 4 5]

Arithmetic UFuncs:
x + 5  = [ 6  7  8  9 10]
x - 2  = [-1  0  1  2  3]
x * 3  = [ 3  6  9 12 15]
x / 2  = [0.5 1.  1.5 2.  2.5]
x // 2 = [0 1 1 2 2]  ← floor division
x ** 2 = [ 1  4  9 16 25]  ← power
x % 2  = [1 0 1 0 1]   ← modulo


In [6]:
# Operations between two arrays (element-wise)
a = np.array([1, 2, 3, 4, 5])
b = np.array([5, 4, 3, 2, 1])

print(f"a = {a}")
print(f"b = {b}")
print(f"a + b = {a + b}")
print(f"a * b = {a * b}")
print(f"a / b = {a / b}")

a = [1 2 3 4 5]
b = [5 4 3 2 1]
a + b = [6 6 6 6 6]
a * b = [5 8 9 8 5]
a / b = [0.2 0.5 1.  2.  5. ]


In [7]:
# Works on multidimensional arrays too!
matrix = np.arange(9).reshape(3, 3)
print("Matrix:")
print(matrix)
print("\n2 ** matrix (element-wise power):")
print(2 ** matrix)

Matrix:
[[0 1 2]
 [3 4 5]
 [6 7 8]]

2 ** matrix (element-wise power):
[[  1   2   4]
 [  8  16  32]
 [ 64 128 256]]


###  UFuncs Quick Reference

| Operator | UFunc | Description |
|----------|-------|-------------|
| `+` | `np.add` | Addition |
| `-` | `np.subtract` | Subtraction |
| `*` | `np.multiply` | Multiplication |
| `/` | `np.divide` | Division |
| `//` | `np.floor_divide` | Floor division |
| `**` | `np.power` | Exponentiation |
| `%` | `np.mod` | Modulo |
| `abs()` | `np.absolute` | Absolute value |

### 1.4 Mathematical Functions

In [8]:
# Absolute value
x = np.array([-3, -2, -1, 0, 1, 2, 3])
print(f"x = {x}")
print(f"abs(x) = {np.abs(x)}")

x = [-3 -2 -1  0  1  2  3]
abs(x) = [3 2 1 0 1 2 3]


In [9]:
# Trigonometric functions (useful for signal processing, rotations)
angles = np.array([0, np.pi/6, np.pi/4, np.pi/3, np.pi/2])
print("Angles (radians):", angles)
print("sin(angles):", np.round(np.sin(angles), 3))
print("cos(angles):", np.round(np.cos(angles), 3))

Angles (radians): [0.         0.52359878 0.78539816 1.04719755 1.57079633]
sin(angles): [0.    0.5   0.707 0.866 1.   ]
cos(angles): [1.    0.866 0.707 0.5   0.   ]


In [10]:
# Exponentials and logarithms (used in ML loss functions, probability)
x = np.array([1, 2, 3])
print(f"x = {x}")
print(f"e^x = {np.exp(x)}")
print(f"ln(x) = {np.log(x)}")
print(f"log10(x) = {np.log10(x)}")

x = [1 2 3]
e^x = [ 2.71828183  7.3890561  20.08553692]
ln(x) = [0.         0.69314718 1.09861229]
log10(x) = [0.         0.30103    0.47712125]


In [11]:
arr1 = np.random.rand(5,5)
arr2 = np.random.rand(5,5)

In [12]:
# matrix multiplication

print(arr1.dot(arr2))
# or
print()
print(np.dot(arr1, arr2))
# or
print()
print(arr1 @ arr2)

[[0.87236969 1.11625461 1.50035348 1.29838129 1.33279559]
 [1.11031285 2.20861172 2.32498128 2.10938449 2.23302423]
 [1.06106754 1.07455034 1.50180003 1.30184452 1.52954539]
 [1.4009696  1.84790816 2.29800886 2.17770181 2.33663709]
 [1.38566455 1.86521396 2.21325239 2.2397856  2.3890266 ]]

[[0.87236969 1.11625461 1.50035348 1.29838129 1.33279559]
 [1.11031285 2.20861172 2.32498128 2.10938449 2.23302423]
 [1.06106754 1.07455034 1.50180003 1.30184452 1.52954539]
 [1.4009696  1.84790816 2.29800886 2.17770181 2.33663709]
 [1.38566455 1.86521396 2.21325239 2.2397856  2.3890266 ]]

[[0.87236969 1.11625461 1.50035348 1.29838129 1.33279559]
 [1.11031285 2.20861172 2.32498128 2.10938449 2.23302423]
 [1.06106754 1.07455034 1.50180003 1.30184452 1.52954539]
 [1.4009696  1.84790816 2.29800886 2.17770181 2.33663709]
 [1.38566455 1.86521396 2.21325239 2.2397856  2.3890266 ]]


In [13]:
import numpy as np

In [14]:
# calculate the inverse/psedo-inverse of a matrix
arr = np.random.rand(5,5)

In [15]:
arr

array([[0.51933599, 0.9126035 , 0.92916653, 0.92547808, 0.60075956],
       [0.32342621, 0.61867631, 0.32914414, 0.58434288, 0.9630125 ],
       [0.64968845, 0.30085624, 0.39338519, 0.04677586, 0.2809042 ],
       [0.2707601 , 0.60178947, 0.88910395, 0.15819783, 0.1844475 ],
       [0.76806204, 0.47609545, 0.82589209, 0.98269829, 0.99204331]])

In [16]:
# compute the inverse of a matrix
print(np.linalg.inv(arr))

[[ 0.60058564 -0.6239347   1.97836155 -1.1802465  -0.09877312]
 [ 1.57248026  0.91067334  0.84481958 -0.5153154  -1.97968887]
 [-1.15329313 -0.55677368 -0.95631228  1.88190159  1.15978   ]
 [ 1.45453514 -0.89542656 -0.67918483 -1.27638413  0.41801885]
 [-1.70034035  1.39653405 -0.46819998  0.85872736  0.65495651]]


---

## Chapter 2: Aggregations — Summarizing Your Data

###  Why Should I Care?

Aggregations are the **first thing** you do in Exploratory Data Analysis (EDA):
- What's the average sale amount?
- What's the maximum temperature recorded?
- What's the total revenue per region?

**Key Insight**: Aggregations can work on the **entire array** OR along a **specific axis** (rows vs. columns).

### 2.1 Basic Aggregations

In [17]:
import numpy as np
rng = np.random.default_rng(seed=42)

# Sample data: 100 random values (think: 100 sales transactions)
data = rng.random(100) * 1000  # Sales between 0-1000
print(f"First 10 values: {data[:10].round(2)}")

First 10 values: [773.96 438.88 858.6  697.37  94.18 975.62 761.14 786.06 128.11 450.39]


In [18]:
# The Big 4 Aggregations
print("=" * 40)
print("Sales Data Summary (100 transactions)")
print("=" * 40)
print(f"Total (sum):   ${np.sum(data):,.2f}")
print(f"Average (mean): ${np.mean(data):,.2f}")
print(f"Minimum:        ${np.min(data):,.2f}")
print(f"Maximum:        ${np.max(data):,.2f}")

Sales Data Summary (100 transactions)
Total (sum):   $48,671.85
Average (mean): $486.72
Minimum:        $7.36
Maximum:        $975.62


In [19]:
# Additional useful aggregations
print(f"Standard Deviation: ${np.std(data):,.2f}")
print(f"Median:             ${np.median(data):,.2f}")
print(f"25th percentile:    ${np.percentile(data, 25):,.2f}")
print(f"75th percentile:    ${np.percentile(data, 75):,.2f}")

Standard Deviation: $272.25
Median:             $468.14
25th percentile:    $232.26
75th percentile:    $709.46


### 2.2 NumPy vs. Python Built-ins: Speed Matters!

In [20]:
big_array = rng.random(1_000_000)

print("Python's built-in sum():")
%timeit sum(big_array)

print("\nNumPy's np.sum():")
%timeit np.sum(big_array)

print("\n→ NumPy is ~100x faster!")

Python's built-in sum():
170 ms ± 3.46 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

NumPy's np.sum():
1.27 ms ± 42.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

→ NumPy is ~100x faster!


### 2.3  The Key Concept: Aggregating Along Axes

This is where NumPy becomes incredibly powerful. You can aggregate:
- **Entire array**: `np.sum(arr)` → single number
- **Along rows** (axis=1): `np.sum(arr, axis=1)` → one value per row
- **Along columns** (axis=0): `np.sum(arr, axis=0)` → one value per column

```
Understanding axis parameter:

Original Matrix (3×4):          axis=0 (collapse rows)    axis=1 (collapse columns)
┌────┬────┬────┬────┐           ┌────┬────┬────┬────┐     ┌────┐
│ a  │ b  │ c  │ d  │           │    │    │    │    │     │ ■  │ sum of row 0
├────┼────┼────┼────┤     →     │ ■  │ ■  │ ■  │ ■  │     ├────┤
│ e  │ f  │ g  │ h  │           sum  sum  sum  sum       │ ■  │ sum of row 1
├────┼────┼────┼────┤           col0 col1 col2 col3      ├────┤
│ i  │ j  │ k  │ l  │                                     │ ■  │ sum of row 2
└────┴────┴────┴────┘           Result: shape (4,)        └────┘
                                                          Result: shape (3,)
```

In [21]:
# Create sample data: Sales by Region (rows) and Quarter (columns)
# 3 regions, 4 quarters
sales = np.array([
    [100, 120, 140, 160],  # Region A: Q1-Q4
    [200, 180, 220, 240],  # Region B: Q1-Q4
    [150, 170, 190, 210]   # Region C: Q1-Q4
])

print("Sales Data (rows=regions, cols=quarters):")
print("         Q1   Q2   Q3   Q4")
print(f"Region A: {sales[0]}")
print(f"Region B: {sales[1]}")
print(f"Region C: {sales[2]}")

Sales Data (rows=regions, cols=quarters):
         Q1   Q2   Q3   Q4
Region A: [100 120 140 160]
Region B: [200 180 220 240]
Region C: [150 170 190 210]


In [22]:
# Total of ALL sales
print(f"Total sales (all regions, all quarters): ${np.sum(sales):,}")

Total sales (all regions, all quarters): $2,080


In [23]:
# Sum along axis=0 (collapse rows) → Total per QUARTER
quarterly_totals = np.sum(sales, axis=0)
print(f"Total per quarter (axis=0): {quarterly_totals}")
print(f"  Q1: ${quarterly_totals[0]}  Q2: ${quarterly_totals[1]}  Q3: ${quarterly_totals[2]}  Q4: ${quarterly_totals[3]}")

Total per quarter (axis=0): [450 470 550 610]
  Q1: $450  Q2: $470  Q3: $550  Q4: $610


In [24]:
# Sum along axis=1 (collapse columns) → Total per REGION
regional_totals = np.sum(sales, axis=1)
print(f"Total per region (axis=1): {regional_totals}")
print(f"  Region A: ${regional_totals[0]}  Region B: ${regional_totals[1]}  Region C: ${regional_totals[2]}")

Total per region (axis=1): [520 840 720]
  Region A: $520  Region B: $840  Region C: $720


In [25]:
# More axis examples
print("\n=== More Aggregation Examples ===")
print(f"\nAverage sales per quarter: {np.mean(sales, axis=0)}")
print(f"Average sales per region:  {np.mean(sales, axis=1)}")
print(f"\nBest quarter per region:   {np.max(sales, axis=1)}")
print(f"Worst quarter per region:  {np.min(sales, axis=1)}")


=== More Aggregation Examples ===

Average sales per quarter: [150.         156.66666667 183.33333333 203.33333333]
Average sales per region:  [130. 210. 180.]

Best quarter per region:   [160 240 210]
Worst quarter per region:  [100 180 150]


###  Aggregation Functions Quick Reference

| Function | NaN-safe version | Description |
|----------|------------------|-------------|
| `np.sum` | `np.nansum` | Sum of elements |
| `np.mean` | `np.nanmean` | Average |
| `np.std` | `np.nanstd` | Standard deviation |
| `np.var` | `np.nanvar` | Variance |
| `np.min` | `np.nanmin` | Minimum value |
| `np.max` | `np.nanmax` | Maximum value |
| `np.argmin` | `np.nanargmin` | Index of minimum |
| `np.argmax` | `np.nanargmax` | Index of maximum |
| `np.median` | `np.nanmedian` | Median value |
| `np.percentile` | `np.nanpercentile` | Percentile |

**Pro Tip**: Use `nan*` versions when your data might contain NaN (missing values)!

---

## Chapter 3: Broadcasting — The Magic of NumPy

###  Why Should I Care?

Broadcasting lets you do math on arrays of **different shapes**. This is essential for:

| Task | What Broadcasting Enables |
|------|---------------------------|
| **Feature Normalization** | Subtract mean from each column: `data - data.mean(axis=0)` |
| **Standardization** | `(data - mean) / std` for each feature |
| **Adding bias** | Add different bias to each feature column |
| **Image processing** | Add color filter to all pixels |

> **This is often the hardest concept for beginners**, but once it clicks, you'll feel like a wizard! 🧙‍♂️

### 3.1 Broadcasting Basics: Start Simple

In [26]:
# Simple case: scalar + array
a = np.array([0, 1, 2])
print(f"a = {a}")
print(f"a + 5 = {a + 5}")

# What's really happening:
# NumPy "broadcasts" 5 to [5, 5, 5]
# Then adds: [0, 1, 2] + [5, 5, 5] = [5, 6, 7]

a = [0 1 2]
a + 5 = [5 6 7]


In [27]:
# 1D array + 2D array
M = np.ones((3, 3))
a = np.array([0, 1, 2])

print("Matrix M (3×3):")
print(M)
print(f"\nArray a: {a}")
print("\nM + a:")
print(M + a)  # 'a' is broadcast across each row!

Matrix M (3×3):
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]

Array a: [0 1 2]

M + a:
[[1. 2. 3.]
 [1. 2. 3.]
 [1. 2. 3.]]


### 3.2 Visualizing Broadcasting

```
Broadcasting example: (3,3) matrix + (3,) array

Matrix M:           Array a:           Result M + a:
┌───┬───┬───┐       ┌───┬───┬───┐     ┌───┬───┬───┐
│ 1 │ 1 │ 1 │       │ 0 │ 1 │ 2 │     │ 1 │ 2 │ 3 │
├───┼───┼───┤   +   │   │   │   │  =  ├───┼───┼───┤
│ 1 │ 1 │ 1 │       │ 0 │ 1 │ 2 │     │ 1 │ 2 │ 3 │
├───┼───┼───┤       │   │   │   │     ├───┼───┼───┤
│ 1 │ 1 │ 1 │       │ 0 │ 1 │ 2 │     │ 1 │ 2 │ 3 │
└───┴───┴───┘       └───┴───┴───┘     └───┴───┴───┘
(3, 3)              (3,) → (3, 3)      (3, 3)
                    ↑ broadcast!
```

### 3.3 The Three Rules of Broadcasting

NumPy compares shapes **element-wise** from right to left:

| Rule | Description |
|------|-------------|
| **Rule 1** | If arrays have different `ndim`, pad the smaller shape with 1s on the **left** |
| **Rule 2** | If shapes differ in a dimension, the array with size 1 is **stretched** to match |
| **Rule 3** | If sizes differ and neither is 1, raise an **error** |

In [28]:
# Example: Broadcasting step by step
a = np.arange(3)              # shape: (3,)
b = np.arange(3)[:, np.newaxis]  # shape: (3, 1)

print(f"a (row vector): {a}")
print(f"a.shape: {a.shape}")
print(f"\nb (column vector):")
print(b)
print(f"b.shape: {b.shape}")

a (row vector): [0 1 2]
a.shape: (3,)

b (column vector):
[[0]
 [1]
 [2]]
b.shape: (3, 1)


In [29]:
# Now add them!
result = a + b
print("a + b:")
print(result)
print(f"\nResult shape: {result.shape}")

a + b:
[[0 1 2]
 [1 2 3]
 [2 3 4]]

Result shape: (3, 3)


```
Broadcasting: (3,) + (3, 1) → (3, 3)

Step 1: Pad shapes to same length
        a: (3,)   → (1, 3)  [add 1 on left]
        b: (3, 1) → (3, 1)  [no change]

Step 2: Stretch dimensions with size 1
        a: (1, 3) → (3, 3)  [stretch first dim]
        b: (3, 1) → (3, 3)  [stretch second dim]

Result: (3, 3)

    a (stretched):      b (stretched):      a + b:
    ┌───┬───┬───┐       ┌───┬───┬───┐      ┌───┬───┬───┐
    │ 0 │ 1 │ 2 │       │ 0 │ 0 │ 0 │      │ 0 │ 1 │ 2 │
    ├───┼───┼───┤   +   ├───┼───┼───┤  =   ├───┼───┼───┤
    │ 0 │ 1 │ 2 │       │ 1 │ 1 │ 1 │      │ 1 │ 2 │ 3 │
    ├───┼───┼───┤       ├───┼───┼───┤      ├───┼───┼───┤
    │ 0 │ 1 │ 2 │       │ 2 │ 2 │ 2 │      │ 2 │ 3 │ 4 │
    └───┴───┴───┘       └───┴───┴───┘      └───┴───┴───┘
```

### 3.4  Practical Application: Centering Data (Feature Normalization)

This is the **#1 use case** for broadcasting in data science!

**Problem**: Given a dataset where each column is a feature, subtract the mean of each column from all values in that column.

In [30]:
# Sample dataset: 4 samples, 3 features
# Think: 4 houses with features [sqft, bedrooms, price]
data = np.array([
    [1000, 2, 200000],
    [1500, 3, 300000],
    [2000, 4, 400000],
    [2500, 5, 500000]
], dtype=float)

print("Original data (4 samples × 3 features):")
print("       sqft  beds    price")
print(data)

Original data (4 samples × 3 features):
       sqft  beds    price
[[1.0e+03 2.0e+00 2.0e+05]
 [1.5e+03 3.0e+00 3.0e+05]
 [2.0e+03 4.0e+00 4.0e+05]
 [2.5e+03 5.0e+00 5.0e+05]]


In [31]:
# Step 1: Calculate mean of each column (feature)
column_means = np.mean(data, axis=0)
print(f"Column means: {column_means}")
print(f"  Mean sqft: {column_means[0]}, Mean beds: {column_means[1]}, Mean price: {column_means[2]}")

Column means: [1.75e+03 3.50e+00 3.50e+05]
  Mean sqft: 1750.0, Mean beds: 3.5, Mean price: 350000.0


In [32]:
# Step 2: Subtract mean from each column (BROADCASTING!)
centered_data = data - column_means

print("Centered data (mean = 0 for each column):")
print(centered_data)
print(f"\nVerify: new column means = {np.mean(centered_data, axis=0)}")

Centered data (mean = 0 for each column):
[[-7.5e+02 -1.5e+00 -1.5e+05]
 [-2.5e+02 -5.0e-01 -5.0e+04]
 [ 2.5e+02  5.0e-01  5.0e+04]
 [ 7.5e+02  1.5e+00  1.5e+05]]

Verify: new column means = [0. 0. 0.]


```
How broadcasting worked:

data.shape:         (4, 3)
column_means.shape: (3,) → (1, 3) → (4, 3)  [broadcast!]

data:               column_means:           centered:
┌──────┬────┬───────┐   ┌──────┬───┬───────┐   ┌──────┬────┬────────┐
│ 1000 │ 2  │200000 │   │ 1750 │3.5│350000 │   │ -750 │-1.5│-150000 │
├──────┼────┼───────┤   │      │   │       │   ├──────┼────┼────────┤
│ 1500 │ 3  │300000 │ - │ 1750 │3.5│350000 │ = │ -250 │-0.5│ -50000 │
├──────┼────┼───────┤   │      │   │       │   ├──────┼────┼────────┤
│ 2000 │ 4  │400000 │   │ 1750 │3.5│350000 │   │  250 │ 0.5│  50000 │
├──────┼────┼───────┤   │      │   │       │   ├──────┼────┼────────┤
│ 2500 │ 5  │500000 │   │ 1750 │3.5│350000 │   │  750 │ 1.5│ 150000 │
└──────┴────┴───────┘   └──────┴───┴───────┘   └──────┴────┴────────┘
```

In [33]:
# Full standardization (z-score): (x - mean) / std
column_stds = np.std(data, axis=0)
standardized_data = (data - column_means) / column_stds

print("Standardized data (mean=0, std=1 for each column):")
print(standardized_data.round(3))
print(f"\nNew means: {np.mean(standardized_data, axis=0).round(10)}")
print(f"New stds:  {np.std(standardized_data, axis=0)}")

Standardized data (mean=0, std=1 for each column):
[[-1.342 -1.342 -1.342]
 [-0.447 -0.447 -0.447]
 [ 0.447  0.447  0.447]
 [ 1.342  1.342  1.342]]

New means: [ 0.  0. -0.]
New stds:  [1. 1. 1.]


### 3.5 When Broadcasting Fails

In [34]:
# This will ERROR: shapes (3,) and (4,) can't broadcast
a = np.array([1, 2, 3])     # shape (3,)
b = np.array([1, 2, 3, 4])  # shape (4,)

try:
    result = a + b
except ValueError as e:
    print(f"Error: {e}")
    print("\nNeither shape is 1, so they can't be broadcast!")

Error: operands could not be broadcast together with shapes (3,) (4,) 

Neither shape is 1, so they can't be broadcast!


###  Broadcasting Quick Reference

| Shape A | Shape B | Result | Works? |
|---------|---------|--------|--------|
| `(3,)` | scalar | `(3,)` | yes |
| `(3, 4)` | `(4,)` | `(3, 4)` | yes |
| `(3, 4)` | `(3, 1)` | `(3, 4)` | yes |
| `(3, 1)` | `(1, 4)` | `(3, 4)` | yes |
| `(3,)` | `(4,)` | Error | no |
| `(3, 4)` | `(3,)` | Error | no |

---

## Chapter 4: Boolean Masking — Filtering Like a Pro

###  Why Should I Care?

Boolean masking lets you answer questions like:
- "Give me all sales above $500"
- "Find all days where temperature was above 80°F AND it was raining"
- "Count how many values are negative"

**This is how you filter data in NumPy (and Pandas)!**

### 4.1 Comparison Operators Create Boolean Arrays

In [35]:
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(f"x = {x}")
print()
print(f"x < 5:  {x < 5}   ← Boolean array!")
print(f"x > 5:  {x > 5}")
print(f"x == 5: {x == 5}")
print(f"x != 5: {x != 5}")

x = [ 1  2  3  4  5  6  7  8  9 10]

x < 5:  [ True  True  True  True False False False False False False]   ← Boolean array!
x > 5:  [False False False False False  True  True  True  True  True]
x == 5: [False False False False  True False False False False False]
x != 5: [ True  True  True  True False  True  True  True  True  True]


In [36]:
# Works on 2D arrays too!
rng = np.random.default_rng(seed=1701)
matrix = rng.integers(0, 10, size=(3, 4))

print("Matrix:")
print(matrix)
print("\nmatrix < 5:")
print(matrix < 5)

Matrix:
[[9 4 0 3]
 [8 6 3 1]
 [3 7 4 0]]

matrix < 5:
[[False  True  True  True]
 [False False  True  True]
 [ True False  True  True]]


### 4.2 Counting with Boolean Arrays

In [37]:
# How many values are less than 5?
print(f"Matrix:\n{matrix}")
print(f"\nHow many values < 5? {np.sum(matrix < 5)}")
print("  (True=1, False=0, so sum counts True values)")

# Or use count_nonzero
print(f"\nUsing count_nonzero: {np.count_nonzero(matrix < 5)}")

Matrix:
[[9 4 0 3]
 [8 6 3 1]
 [3 7 4 0]]

How many values < 5? 8
  (True=1, False=0, so sum counts True values)

Using count_nonzero: 8


In [38]:
# Count per row
print(f"Values < 5 per row: {np.sum(matrix < 5, axis=1)}")

Values < 5 per row: [3 2 3]


In [39]:
# Quick checks: any() and all()
print(f"Any value > 8?  {np.any(matrix > 8)}")
print(f"All values < 20? {np.all(matrix < 20)}")
print(f"All values > 5?  {np.all(matrix > 5)}")

Any value > 8?  True
All values < 20? True
All values > 5?  False


### 4.3 Boolean Operators: Combining Conditions

 **Important**: Use `&` (and), `|` (or), `~` (not) instead of Python's `and`, `or`, `not`!

| Operator | Meaning | Example |
|----------|---------|--------|
| `&` | AND | `(x > 2) & (x < 7)` |
| `\|` | OR | `(x < 2) \| (x > 7)` |
| `~` | NOT | `~(x == 5)` |

In [40]:
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Values between 3 and 7 (inclusive)
mask = (x >= 3) & (x <= 7)
print(f"x = {x}")
print(f"(x >= 3) & (x <= 7): {mask}")

x = [ 1  2  3  4  5  6  7  8  9 10]
(x >= 3) & (x <= 7): [False False  True  True  True  True  True False False False]


In [41]:
# Values less than 3 OR greater than 7
mask = (x < 3) | (x > 7)
print(f"(x < 3) | (x > 7): {mask}")

(x < 3) | (x > 7): [ True  True False False False False False  True  True  True]


In [42]:
# NOT equal to 5
mask = ~(x == 5)
print(f"~(x == 5): {mask}")

~(x == 5): [ True  True  True  True False  True  True  True  True  True]


### 4.4  Boolean Arrays as Masks: Filtering Data

**The Magic**: Use a boolean array to index into another array!

In [43]:
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(f"x = {x}")

# Create a mask
mask = x < 5
print(f"mask (x < 5) = {mask}")

# Apply the mask - only returns values where mask is True!
filtered = x[mask]
print(f"x[mask] = {filtered}")

x = [ 1  2  3  4  5  6  7  8  9 10]
mask (x < 5) = [ True  True  True  True False False False False False False]
x[mask] = [1 2 3 4]


In [44]:
# More concise: do it in one line
print(f"Values > 5: {x[x > 5]}")
print(f"Values between 3 and 7: {x[(x >= 3) & (x <= 7)]}")
print(f"Even values: {x[x % 2 == 0]}")

Values > 5: [ 6  7  8  9 10]
Values between 3 and 7: [3 4 5 6 7]
Even values: [ 2  4  6  8 10]


### 4.5 Real-World Example: Weather Data Analysis

In [45]:
# Simulate 365 days of rainfall data (mm)
rng = np.random.default_rng(seed=42)

# Many dry days, some light rain, few heavy rain
rainfall_mm = rng.exponential(scale=3, size=365)
rainfall_mm[rng.random(365) < 0.4] = 0  # 40% of days are dry

print(f"Total days: {len(rainfall_mm)}")
print(f"First 10 days: {rainfall_mm[:10].round(1)}")

Total days: 365
First 10 days: [0.  7.  7.2 0.  0.  0.  0.  0.  0.2 3.1]


In [46]:
# Quick stats using boolean masking
print("=" * 40)
print("Weather Analysis (2015)")
print("=" * 40)
print(f"Days without rain:     {np.sum(rainfall_mm == 0)}")
print(f"Days with rain:        {np.sum(rainfall_mm > 0)}")
print(f"Days with heavy rain (>10mm): {np.sum(rainfall_mm > 10)}")
print(f"Light rainy days (0-5mm):     {np.sum((rainfall_mm > 0) & (rainfall_mm < 5))}")

Weather Analysis (2015)
Days without rain:     141
Days with rain:        224
Days with heavy rain (>10mm): 5
Light rainy days (0-5mm):     183


In [47]:
# Now use masking to get actual values
rainy = rainfall_mm > 0

print(f"\nAverage rainfall on rainy days: {np.mean(rainfall_mm[rainy]):.2f} mm")
print(f"Max rainfall: {np.max(rainfall_mm):.2f} mm")


Average rainfall on rainy days: 2.85 mm
Max rainfall: 16.29 mm


In [48]:
# Combine multiple conditions: summer rainy days
days = np.arange(365)
summer = (days >= 172) & (days < 265)  # June 21 to Sep 21

print(f"\nSummer analysis:")
print(f"  Total summer days: {np.sum(summer)}")
print(f"  Summer rainy days: {np.sum(summer & rainy)}")
print(f"  Average summer rainfall: {np.mean(rainfall_mm[summer]):.2f} mm")
print(f"  Average summer rainfall (rainy days only): {np.mean(rainfall_mm[summer & rainy]):.2f} mm")


Summer analysis:
  Total summer days: 93
  Summer rainy days: 63
  Average summer rainfall: 2.06 mm
  Average summer rainfall (rainy days only): 3.04 mm


---

##  Session 2 Summary

### What You Learned

| Chapter | Key Concept | Data Science Application |
|---------|-------------|-------------------------|
| 1 | UFuncs (vectorization) | 100-1000x speedup vs. loops |
| 2 | Aggregations + axis | Calculate stats per row/column |
| 3 | Broadcasting | Normalize features for ML |
| 4 | Boolean masking | Filter and subset your data |

###  Key Takeaways

1. **Never use Python loops** on NumPy arrays — use vectorized operations
2. **axis=0** collapses rows (result: one value per column)
3. **axis=1** collapses columns (result: one value per row)
4. **Broadcasting** lets you do math on different-shaped arrays
5. **Boolean masking** is how you filter data: `arr[arr > 5]`

###  What's Next?

With these NumPy fundamentals, you're ready for:
- **Pandas**: DataFrames are built on NumPy arrays
- **Matplotlib/Seaborn**: Visualization (NumPy arrays are the input)
- **Scikit-learn**: ML models expect NumPy arrays

---

## Practice Exercises

In [54]:
# Exercise 1: Given this sales data (rows=stores, cols=months),
# calculate: (a) total sales per store, (b) average sales per month
sales = np.array([
    [100, 120, 140, 160],
    [200, 180, 220, 240],
    [150, 170, 190, 210]
])
# Your code here:
print(f"Total Salses Per Store = {np.sum(sales,axis=1)}")
print(f"Average Sales Per Month = {np.mean(sales,axis=0)}")

Total Salses Per Store = [520 840 720]
Average Sales Per Month = [150.         156.66666667 183.33333333 203.33333333]


In [57]:
# Exercise 2: Center this data (subtract column means) using broadcasting
data = np.array([
    [10, 100, 1000],
    [20, 200, 2000],
    [30, 300, 3000],
    [40, 400, 4000]
], dtype=float)
# Your code here:
col_means = np.mean(data,axis=0)
result =data - col_means
print(result)

[[  -15.  -150. -1500.]
 [   -5.   -50.  -500.]
 [    5.    50.   500.]
 [   15.   150.  1500.]]


In [69]:
# Exercise 3: From this array, find:
# (a) All values greater than 50
# (b) Count of values between 20 and 80
# (c) Mean of values that are NOT divisible by 10
arr = np.array([10, 25, 30, 45, 50, 65, 70, 85, 90, 100])
# Your code here:
print(arr[arr>50])
print(np.sum((arr>20) & (arr<80)))
print(np.mean(arr[~(arr%10==0)]))

[ 65  70  85  90 100]
6
55.0


In [52]:
# Exercise 4: Given exam scores for 5 students across 3 exams,
# find which students passed ALL exams (score >= 60)
scores = np.array([
    [85, 90, 78],   # Student 0
    [55, 62, 70],   # Student 1
    [92, 88, 95],   # Student 2
    [60, 58, 65],   # Student 3
    [75, 80, 72]    # Student 4
])
# Hint: Use np.all() with axis parameter, then use that as a mask
# Your code here:
